In [49]:
import sys
sys.path.append("..")
import importlib
import clean_data

importlib.reload(clean_data)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path 
from catboost import CatBoostClassifier, Pool 

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, log_loss, f1_score, balanced_accuracy_score, accuracy_score

from clean_data import feature_engineering_days

In [50]:
df = pd.read_csv("../data/austin/raw/aac_intakes.csv")
outcomes = pd.read_csv("../data/austin/raw/aac_intakes_outcomes.csv")

df["datetime"] = pd.to_datetime(df["datetime"])
outcomes["intake_datetime"] = pd.to_datetime(outcomes["intake_datetime"])

df = df.merge(
    outcomes[["animal_id_intake", "intake_datetime", "time_in_shelter_days"]],
    left_on=["animal_id", "datetime"],
    right_on=["animal_id_intake", "intake_datetime"],
    how="inner"
)

df = df[df["animal_type"] == "Dog"].copy()

In [51]:
df = feature_engineering_days(df)

df_dogs = df[df["animal_type"] == "Dog"].copy()
df_dogs = df_dogs.sort_values("intake_datetime").reset_index(drop=True)

bins = [-np.inf, 7, 30, np.inf]
df_dogs["target"] = pd.cut(
    df_dogs["time_in_shelter_days"], bins=bins, labels=[0, 1, 2]
).astype(int)

In [52]:
drop_cols = [
    "target", "time_in_shelter_days", "time_in_shelter",
    "outcome_type", "outcome_subtype", "sex_upon_outcome",
    "intake_datetime", "animal_type", "young_and_cat", "datetime", "animal_id_intake"
]

X = df_dogs.drop(columns=drop_cols, errors="ignore")
y = df_dogs["target"]

#categorical features which will be available
cat_features = X.select_dtypes(include=["object", "category"]).columns.tolist()
X[cat_features] = X[cat_features].fillna("Unknown").astype(str)

#70/15/15
n = len(X)
i1, i2 = int(n * 0.70), int(n * 0.85)

X_train, X_val, X_test = X.iloc[:i1], X.iloc[i1:i2], X.iloc[i2:]
y_train, y_val, y_test = y.iloc[:i1], y.iloc[i1:i2], y.iloc[i2:]

print("Split:", len(X_train), len(X_val), len(X_test))

Split: 31434 6736 6737


In [53]:
model = CatBoostClassifier(
    iterations=1200, learning_rate=0.03, depth=6,
    loss_function="MultiClass", random_seed=42, verbose=100
)

model.fit(
    X_train, y_train,
    cat_features=cat_features,
    eval_set=(X_val, y_val),
    early_stopping_rounds=100,
    use_best_model=True
)

0:	learn: 1.0842314	test: 1.0820652	best: 1.0820652 (0)	total: 134ms	remaining: 2m 41s
100:	learn: 0.7837494	test: 0.7442222	best: 0.7442222 (100)	total: 12.6s	remaining: 2m 17s
200:	learn: 0.7550856	test: 0.7260175	best: 0.7260175 (200)	total: 22s	remaining: 1m 49s
300:	learn: 0.7438117	test: 0.7209400	best: 0.7209400 (300)	total: 30.9s	remaining: 1m 32s
400:	learn: 0.7332840	test: 0.7165891	best: 0.7165891 (400)	total: 39.9s	remaining: 1m 19s
500:	learn: 0.7253908	test: 0.7139143	best: 0.7139143 (500)	total: 49.1s	remaining: 1m 8s
600:	learn: 0.7198868	test: 0.7130944	best: 0.7130575 (597)	total: 58.2s	remaining: 58s
700:	learn: 0.7148205	test: 0.7121395	best: 0.7120729 (696)	total: 1m 7s	remaining: 47.9s
800:	learn: 0.7092562	test: 0.7113644	best: 0.7113644 (800)	total: 1m 16s	remaining: 38.2s
900:	learn: 0.7047327	test: 0.7108246	best: 0.7108138 (895)	total: 1m 26s	remaining: 28.6s
1000:	learn: 0.7001251	test: 0.7104510	best: 0.7103598 (972)	total: 1m 36s	remaining: 19.2s
1100:	lea

CatBoostClassifier(depth=6, iterations=1200, learning_rate=0.03, loss_function='MultiClass', random_seed=42, verbose=100)

In [54]:
X_train.columns

Index(['breed', 'color', 'found_location', 'intake_condition', 'intake_type',
       'name', 'sex', 'intact', 'is_mixed_breed', 'primary_breed',
       'is_mixed_color', 'primary_color', 'has_name', 'name_length',
       'age_days', 'day_of_week', 'month', 'year', 'is_weekend'],
      dtype='object')

# Evaluation


In [55]:
probs = model.predict_proba(X_test)
y_pred = model.classes_[np.argmax(probs, axis=1)]

#Log Loss
ll = log_loss(y_test, probs, labels=model.classes_)

#Multiclass Brier Score
y_onehot = (
    np.asarray(y_test)[:, None]
    == np.asarray(model.classes_)[None, :]
).astype(int)

brier = np.mean(np.sum((probs - y_onehot) ** 2, axis=1))

#Macro F1
f1 = f1_score(y_test, y_pred, average="macro")

#Balanced Accuracy
bal_acc = balanced_accuracy_score(y_test, y_pred)

print(f"Log Loss:          {ll:.4f}")
print(f"Brier Score:       {brier:.4f}")
print(f"Macro F1:          {f1:.4f}")
print(f"Balanced Accuracy: {bal_acc:.4f}")

print("\nClassification report:")
print(classification_report(y_test, y_pred))

Log Loss:          0.7241
Brier Score:       0.4178
Macro F1:          0.4957
Balanced Accuracy: 0.4803

Classification report:
              precision    recall  f1-score   support

           0       0.75      0.90      0.82      4361
           1       0.51      0.34      0.40      1698
           2       0.38      0.21      0.27       678

    accuracy                           0.69      6737
   macro avg       0.54      0.48      0.50      6737
weighted avg       0.65      0.69      0.66      6737



In [56]:
from sklearn.metrics import precision_recall_curve, average_precision_score

classes = list(model.classes_)
i2 = classes.index(2)

p2_val = model.predict_proba(X_val)[:, i2]
p2_test = model.predict_proba(X_test)[:, i2]

# Wybór progu na walidacji: najwyższa precyzja przy recall >= 0.6
prec, rec, thr = precision_recall_curve((y_val == 2).astype(int), p2_val)
ok = rec[:-1] >= 0.8
threshold = thr[ok][np.argmax(prec[:-1][ok])]
print(f"Próg dla klasy >30 dni: {threshold:.3f}")

# Ocena na teście
flag = p2_test >= threshold
y_long = (y_test == 2).astype(int)
tp = np.sum(flag & (y_long == 1))
print(f"Oznaczone: {flag.mean():.1%} zwierząt")
print(f"Precyzja: {tp / max(flag.sum(), 1):.3f}")
print(f"Recall:   {tp / y_long.sum():.3f}")
print(f"PR-AUC (klasa >30): {average_precision_score(y_long, p2_test):.3f}")

Próg dla klasy >30 dni: 0.102
Oznaczone: 35.6% zwierząt
Precyzja: 0.199
Recall:   0.704
PR-AUC (klasa >30): 0.293


In [57]:
def predict_dog(model, dog):
    probs = model.predict_proba(dog)[0]

    return pd.DataFrame({
        "Czas pobytu": ["≤7 dni", "7–30 dni", ">30 dni"],
        "Prawdopodobieństwo (%)": [
            round(probs[list(model.classes_).index(i)] * 100, 2)
            for i in [0, 1, 2]
        ]
    })


In [58]:
dog = X_test.iloc[[0]]
print(df_dogs.loc[dog.index[0], "time_in_shelter_days"])
display(predict_dog(model, dog))

1.16875


,Czas pobytu,Prawdopodobieństwo (%)
0,≤7 dni,87.23
1,7–30 dni,11.36
2,>30 dni,1.41


In [59]:
dog = X_test.iloc[[30]]

print(df_dogs.loc[dog.index[0], "time_in_shelter_days"])

display(predict_dog(model, dog))

0.1284722222222222


,Czas pobytu,Prawdopodobieństwo (%)
0,≤7 dni,84.98
1,7–30 dni,13.19
2,>30 dni,1.84


In [60]:
long_stay = X_test[df_dogs.loc[X_test.index, "time_in_shelter_days"] > 60]

dog = long_stay.iloc[[0]]

print(df_dogs.loc[dog.index[0], "time_in_shelter_days"])
display(predict_dog(model, dog))


64.74027777777778


,Czas pobytu,Prawdopodobieństwo (%)
0,≤7 dni,48.50
1,7–30 dni,34.76
2,>30 dni,16.74


In [ ]:
# import shap

# explainer = shap.TreeExplainer(model)
# X_sample = X_test.sample(n=min(500, len(X_test)), random_state=42)
# shap_values = explainer(X_sample)

# print(shap_values.values.shape)

: 

In [ ]:
# class_idx = list(model.classes_).index(2)
# shap_long = shap_values[:, :, class_idx]
# shap.plots.beeswarm(shap_long, max_display=15)

In [ ]:
# dog = X_test.iloc[[0]]
# display(predict_dog(model, dog))

# dog_shap = explainer(dog)

# shap.plots.waterfall(
#     dog_shap[0, :, class_idx],
#     max_display=15
# )